# Makilala FoS Pipeline — v9

**Closed-form pseudo-static slope stability applied to Makilala, North Cotabato**
Cubio, Dizon, Daag, Grutas, Villarias, Segales & Arcede

This notebook produces every quantitative result in the manuscript: the Factor
of Safety, critical seismic coefficient $k_c$ and Form B sensitivity bound over
the evaluated domain; the cohesion × depth × threshold sweep that fixes the
operating point; the validation at pixel, polygon, intensity and spatially
blocked scales; and the figures.

---

## The authoritative configuration

Everything reported in the manuscript comes from **one** configuration, pinned
in Part I §7. **All numeric results below are superseded by this run:** v9
corrects the raster registration in §3, which changes every quantity that
depends on slope, soil depth or ground motion.


| | |
|---|---|
| cohesion | `c_plus10` — correlation baseline $+10$ kPa |
| depth cap | `z_cap_5m` |
| threshold | $\theta^{*} = 1.05$, the Youden optimum for that configuration |
| scored domain | **re-measured by this run** — v8 reported 3,870,885 px, but that figure came from a misregistered soil-depth raster |

The other eleven configurations in the sweep exist to show how $\theta^{*}$
moves with cohesion and how little AUC moves with either. They are not
alternative results.

Why `c_plus10` is primary is a substantive choice, not a fitted one: it is the
configuration at which the model's own failure criterion ($\mathrm{FoS} = 1$)
coincides with the empirical optimum. At the correlation baseline the optimum
sits at 0.65, well below unity, which indicates the cohesion input is too low
rather than that the model discriminates better there — AUC across all twelve
configurations spans only 0.752–0.779.

## How to run

Run Part I top to bottom. Cell for cell it is order-dependent; §7 in particular
needs everything above it. The sweep in §6 is the slow step — twelve
configurations over 21.1 million pixels, a few minutes.

Part II is independent of itself but not of Part I: every diagnostic there
assumes Part I has completed in the same kernel. Sections may be run in any
order or skipped.

**Memory.** §6 retains all twelve configurations, each holding three full-grid
float64 arrays — roughly 550 MB per configuration, about 6.6 GB resident. If
later cells fail with a memory error, free the eleven that are not primary
*after* §7 has run:

```python
import gc
keep = {BEST_KEY: cohesion_results[BEST_KEY]}
cohesion_results.clear(); cohesion_results.update(keep)
for _v in ('grid', 'all_results'):
    globals().pop(_v, None)
gc.collect()
```

That makes Part II §A and §C unrunnable until §6 is re-run, since both read
non-primary configurations.

## Data provenance

| layer | source | native resolution |
|---|---|---|
| slope | ML-PREP DEM | **5 m** |
| geotechnical (8 parameters) | EBK from 37 field samples, ArcGIS Pro 3.1.0 | ~74 m |
| soil depth | SoilGrids 2.0 | 250 m (229 m reprojected) |
| PGA | PHIVOLCS REDAS, three Oct 2019 events, max-value mosaic | ~555 m |
| inventory | PHIVOLCS, province-wide | vector |

Everything is resampled to a 5 m grid, and that grid is a computational
convenience rather than a claim about information content. Only slope is
natively 5 m. This matters when reading the maps: the framework resolves
terrain, not pixels, which is the same conclusion the blocked validation
reaches independently.

**Coordinate systems differ between the two raster families.** The prediction
rasters (`Mak_*.tif`) are ESRI:102457 (PRS 1992 UTM 51N) in metres; the EBK
standard-error rasters are EPSG:4326 in degrees over a smaller footprint. Cell
§3 aligns the former with `scipy.ndimage.zoom`, which is a pure array rescale
and is correct only because they already share the reference CRS and extent.
Anything read from the `standard_error/` folder must be **reprojected**, not
zoomed — see Part II §F.

**EBK settings** (Segales, ArcGIS Pro 3.1.0): Power semivariogram, no data
transformation, max 100 points per local model, overlap factor 1, 100 simulated
semivariograms, standard circular neighbourhood, 10–15 neighbours, search radius
$5.053\times10^{-2}$ degrees ($\approx5.6$ km). Note that the local-model limit
of 100 exceeds the 37 samples available, so a single local model spans the whole
area and EBK's local adaptivity is not exercised at this sample density.

## Field data

43 samples were collected; 6 were withdrawn for incomplete laboratory
determinations, leaving **37** with a complete index-property set — 22 landslide
and 15 non-landslide sites across 21 barangays. All interpolated fields derive
from those 37. (Part II §E reads all 43, because site proximity to the inventory
does not depend on laboratory completeness.)

---

## Changes in v8

**Corrections**

1. `scored` was rebound in the polygon-accounting cell to a per-polygon array of
   length 3,140, shadowing the pixel mask and crashing the final-maps cell with
   a shape mismatch. The polygon-level array is now `poly_scored`.
2. The validation cell computed ESI confidence intervals against *all* polygons
   at each intensity, though the hit rate is computed over scored pixels and
   only 1,002 of 1,699 polygons contribute any. With the corrected denominator
   **no adjacent ESI pair separates** — see Part I §7.
3. The old validation-overlay cell rebound `mask` to a local figure mask,
   silently changing it for every cell that ran afterwards. That cell is
   superseded and removed.
4. The threshold grid is $[0.40, 2.00]$. The v7 header and §6 markdown both said
   $[0.40, 1.50]$, which never matched the code.
5. Point-level validation uses 37 complete samples of 43 collected. The v7
   header said 41.

**Removed** — the `grid_filled` stash (its comparison is recorded in the
manuscript), two scratch cells, an exploratory read of the superseded
`makilala_41_enriched.xlsx`, the polygon-detection loop that rasterized the full
grid 12,560 times to compute what two `bincount` calls give, and the duplicate
validation overlay.

**Added** — Part II §F, uncertainty propagation for $k_c$.

## Known limitations, carried forward deliberately

- Pooled AUC is optimistic under spatial autocorrelation. The blocked estimate
  in §7 is the honest one, and it is materially lower (0.642 against 0.779).
- `estimate_m` returns a 0.50 fallback where water content is missing or zero,
  so the saturation raster mixes measured and default values. Over the scored
  domain the realised field is constant at 0.6 regardless.
- `np.clip(z_base, 0.1, ...)` maps zero-depth pixels to 0.1 m, where the
  $c'/(\gamma z\cos^2\alpha)$ term makes them confidently stable. This affects
  **3,688 pixels**, 0.1% of the scored domain. `EXCLUDE_ZERO_DEPTH` in §1
  removes them; it defaults to `False` so that this notebook reproduces the
  published figures exactly.

---
# Part I — Pipeline

Order-dependent. Run top to bottom.

## §1 Environment and configuration

In [ ]:
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')
from scipy.ndimage import zoom

import rasterio
from rasterio.features import rasterize
from shapely.geometry import mapping
print(f"✓ rasterio {rasterio.__version__}")

try:
    import geopandas as gpd
    print(f"✓ geopandas {gpd.__version__}")
except ImportError:
    gpd = None; print("⚠ geopandas not found")

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
print(f"✓ matplotlib, numpy")


In [ ]:
INPUT_DIR = os.environ.get("MAKILALA_DIR", "data")
OUTPUT_DIR = os.path.join(INPUT_DIR, "output_maps_v9")

FILE_SLOPE = "Slope_PRSTM51N.tif"; FILE_DEPTH = "Soil Depth.tif"
FILE_PGA = "PGA 2019 EQ Events.tif"; FILE_GAMMA = "Mak_BulkUnitWeight.tif"
FILE_WATER = "Mak_WaterContent.tif"; FILE_PI = "Mak_PlasticityIndex.tif"
FILE_FINES = "Mak_Fines.tif"; FILE_SAND = "Mak_Sand.tif"
FILE_LSI = "LSI_North Cotabato.shp"

REFERENCE_GRID = 'gamma'
ALPHA_MIN = 10       # Minimum slope angle for failure consideration (degrees)
GAMMA_W = 9.81; G_ACC = 9.81; NODATA = -9999.0

# ============================================================================
#  COORDINATE REFERENCE SYSTEMS
#
#  These are the only lines in the notebook that name a CRS. If J.M.B. Segales
#  reports different provenance for the EBK export or the inventory, change
#  them here and rerun; nothing downstream needs editing.
#
#  GRID_CRS is what the Mak_*.tif rasters ACTUALLY are, which is not what all
#  of them say. Mak_Fines, Mak_Sand and Mak_WaterContent are tagged EPSG:32651;
#  Mak_BulkUnitWeight and Mak_PlasticityIndex are tagged ESRI:102457. All five
#  share a byte-identical grid, so at most one tag can be right. Sampling all
#  five at the 38 GPS sites with complete laboratory determinations puts the
#  best fit at +170 m E, -45 m N from the ESRI:102457 reading -- against a
#  PRS92 -> WGS84 datum shift of +167.9 m E, -46.8 m N at the study centroid.
#  The EPSG:32651 tag is the correct one.
#
#  LSI_CRS is what LSI_North Cotabato.shp ACTUALLY is. Its .prj declares
#  ESRI:102457. Under that reading the 25 field sites recorded AT landslide
#  sites sit a median 67.5 m from the nearest polygon; read as EPSG:32651 the
#  median is 21.9 m, and sites within 50 m go from 8/25 to 18/25. Handheld GPS
#  reports WGS 84 by construction, so the GPS is the ground truth here.
#
#  The external layers keep whatever their own files declare, unless overridden
#  below. Slope is PRS 1992 as labelled: it is the layer the other two disagree
#  with, by exactly one datum shift.
# ============================================================================
GRID_CRS = "EPSG:32651"      # WGS 84 / UTM 51N -- the Mak_*.tif reference grid
LSI_CRS = "EPSG:32651"      # WGS 84 / UTM 51N -- the inventory shapefile

# Per-layer overrides for the external rasters. None = trust the file's own CRS.
CRS_OVERRIDE = {
    'slope': None,           # ESRI:102457, verified correct against terrain
    'depth': None,           # SoilGrids -> reprojected by a third party; unverified
    'pga': None,           # PHIVOLCS REDAS; unverified
}

# Zero-depth handling. The SoilGrids raster carries 286 literal zeros which the
# resample spreads across neighbouring pixels. Because z_base is clipped to a
# 0.1 m floor before the validity test, those pixels enter the scored domain at
# 10 cm of soil and are predicted stable almost by construction. Under v8's
# zoom this affected 3,688 px; under v9's reprojection the count differs and is
# reported by §3. Left False for continuity with v8.
EXCLUDE_ZERO_DEPTH = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Grid CRS: {GRID_CRS}   Inventory CRS: {LSI_CRS}")
print(f"Min slope for failure: {ALPHA_MIN}°")


## §2 Correlations and the closed forms

Mohr–Coulomb strength is estimated by correlation, not measured: direct shear
testing was lost when the October 2025 Davao Oriental earthquake red-tagged the
laboratory. `estimate_phi` and `estimate_c` are step functions of interpolated
index properties, and that is consequential — thresholding a conditional-mean
surface is not the mean of the thresholding, and the standard error of
interpolated fines (4.6 pp) is comparable to the class widths (7–8 pp) the
cohesion rule cuts on. Part II §F quantifies what that costs.

`compute_FoS` and `compute_kc` are the manuscript's Eq. (1) and Eq. (3);
`sensitivity_FormB` is the Form B bound, defined only where
$\mathrm{FoS}_{\mathrm{static}} > 1$.

In [ ]:
def estimate_phi(pi, sand, fines):
    phi = np.full_like(pi, 30.0, dtype=np.float64)
    v = (pi > 5) & np.isfinite(pi)
    phi[v] = 35.0 - 0.1 * pi[v]
    no = ~v & np.isfinite(sand)
    phi[no & (sand > 85) & (fines < 10)] = 33.0
    phi[no & (sand > 70) & (fines >= 10) & (fines < 20)] = 31.0
    phi[no & (fines >= 20)] = 28.0
    return np.clip(phi, 20.0, 38.0)

def estimate_c(fines, sand):
    c = np.full_like(fines, 5.0, dtype=np.float64)
    v = np.isfinite(fines) & np.isfinite(sand)
    c[v & (fines < 5)] = 1.0; c[v & (fines >= 5) & (fines < 12)] = 3.0
    c[v & (fines >= 12) & (fines < 20)] = 6.0; c[v & (fines >= 20)] = 10.0
    return c

def estimate_m(water):
    m = np.full_like(water, 0.5, dtype=np.float64)
    v = np.isfinite(water) & (water > 0)
    m[v & (water < 20)] = 0.2; m[v & (water >= 20) & (water < 35)] = 0.4
    m[v & (water >= 35) & (water < 50)] = 0.6; m[v & (water >= 50)] = 0.8
    return m

def compute_FoS(c, phi_deg, alpha_deg, m, gamma, z, k_h):
    phi, alpha = np.radians(phi_deg), np.radians(alpha_deg)
    R = c + (gamma - m * GAMMA_W) * z * np.cos(alpha)**2 * np.tan(phi)
    D = gamma * z * np.sin(alpha) * np.cos(alpha) + k_h * gamma * z * np.cos(alpha)**2
    return np.where(D > 0, R / D, np.nan)

def compute_kc(c, phi_deg, alpha_deg, m, gamma, z):
    phi, alpha = np.radians(phi_deg), np.radians(alpha_deg)
    d = gamma * z * np.cos(alpha)**2
    return np.where(d > 0, c/d + (1-m*GAMMA_W/gamma)*np.tan(phi) - np.tan(alpha), np.nan)

def sensitivity_FormB(fs):
    return np.where(fs > 1, 1.0 - 1.0/fs, np.nan)

print("✓ Functions defined")


## §3 Rasters: load and register

Alignment is by `rasterio.warp.reproject`, which maps each source pixel through
its own coordinate system onto the reference grid. v8 used `scipy.ndimage.zoom`,
a pure array rescale that ignores CRS and bounds, on the stated assumption that
every raster already shared the reference extent. Three do not:

| layer | grid | cell | extent |
|---|---|---|---|
| `Slope_PRSTM51N` | 3716 × 5731 | 5 m | 28.7 × 18.6 km, origin offset |
| `Soil Depth` | 444 × 461 | 229 m | 105.6 × 102.7 km |
| `PGA 2019 EQ Events` | 702 × 701 | 555 m | 389.0 × 389.6 km |

The reference window is 28.5 × 18.5 km, so `zoom` squeezed a 105.6 km raster and
a 389 km raster onto it. Registered properly, depth correlates **−0.245** with its
zoomed self and PGA **0.020**; mean PGA over the municipality is **0.302 g**, not
0.038 g, against an instrumental 0.191 g recorded at KCPA 1 km from the municipal
centre during a *smaller* aftershock.

The five `Mak_*` rasters share the reference grid exactly and are passed through
untouched — reprojecting them would be a no-op at best and, because three of the
five are tagged `EPSG:32651` and two `ESRI:102457`, would silently apply a datum
shift to some and not others. `GRID_CRS` in §0 settles which tag is right; the
files themselves are never re-read for their CRS.

`gamma` defines the reference grid. This cell reports, for every layer, whether
it was passed through or reprojected and from what.


In [ ]:
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS


def load_raster(fp, label=""):
    with rasterio.open(fp) as src:
        d = src.read(1).astype(np.float64); p = src.profile.copy()
        if src.nodata is not None: d[d == src.nodata] = np.nan
        v = np.isfinite(d) & (d != 0)
        print(f"  ✓ {label}: {d.shape} @ {src.res[0]:g} {'m' if src.crs and src.crs.is_projected else 'deg'}"
              f", {src.crs}, valid={v.sum():,}, [{np.nanmin(d[v]):.3f}, {np.nanmax(d[v]):.3f}]")
        return d, p


rasters, profiles = {}, {}
for key, (fn, lb) in {'slope': (FILE_SLOPE,"α"), 'depth': (FILE_DEPTH,"z"),
    'pga': (FILE_PGA,"PGA"), 'gamma': (FILE_GAMMA,"γ"), 'water': (FILE_WATER,"w"),
    'pi': (FILE_PI,"PI"), 'fines': (FILE_FINES,"fines"), 'sand': (FILE_SAND,"sand")}.items():
    fp = os.path.join(INPUT_DIR, fn)
    if os.path.exists(fp):
        d, p = load_raster(fp, lb); rasters[key] = d; profiles[key] = p
    else:
        print(f"  ✗ {lb}: NOT FOUND")

ref_shape = rasters[REFERENCE_GRID].shape
ref_profile = profiles[REFERENCE_GRID]
ref_transform = ref_profile['transform']
ref_crs = CRS.from_string(GRID_CRS)          # §0, NOT the file's own tag
ref_profile = {**ref_profile, 'crs': ref_crs}

print(f"\nReference grid: {ref_shape[0]}×{ref_shape[1]} @ {ref_transform.a:g} m")
print(f"  file says : {profiles[REFERENCE_GRID].get('crs')}")
print(f"  GRID_CRS  : {ref_crs}   <- used")
if str(profiles[REFERENCE_GRID].get('crs')) != str(ref_crs):
    print("  (the file's tag is overridden; see the CRS note in §0)")

# ---------------------------------------------------------------------------
#  Registration. Layers already on the reference grid pass through untouched --
#  never reproject them, because the Mak_* tags are mutually inconsistent and a
#  reprojection would shift some by a datum and leave others alone.
# ---------------------------------------------------------------------------
aligned = {}
for key, data in rasters.items():
    same_grid = (data.shape == ref_shape
                 and profiles[key]['transform'].almost_equals(ref_transform, precision=1e-6))
    if same_grid:
        aligned[key] = data
        print(f"  = {key:<6}: on the reference grid, passed through")
        continue

    src_crs = profiles[key]['crs']
    override = CRS_OVERRIDE.get(key)
    if override is not None:
        src_crs = CRS.from_string(override)
    if src_crs is None:
        raise RuntimeError(f"{key}: no CRS on the file and no CRS_OVERRIDE entry; "
                           f"cannot register it. Set CRS_OVERRIDE['{key}'] in §0.")

    dst = np.full(ref_shape, np.nan, dtype=np.float64)
    reproject(source=data, destination=dst,
              src_transform=profiles[key]['transform'], src_crs=src_crs,
              dst_transform=ref_transform, dst_crs=ref_crs,
              resampling=Resampling.bilinear,
              src_nodata=np.nan, dst_nodata=np.nan)
    aligned[key] = dst
    tag = f" (CRS_OVERRIDE -> {override})" if override else ""
    cov_before = np.isfinite(data).mean(); cov_after = np.isfinite(dst).mean()
    print(f"  ↳ {key:<6}: {str(data.shape):>13} {str(src_crs):<13}{tag} → reprojected, "
          f"coverage {100*cov_after:.1f}% of the grid")

print("\n✓ All layers on the reference grid")

# ---------------------------------------------------------------------------
#  Registration sanity check. Landslides occur on steep ground, so a correctly
#  registered slope raster must show a large mean-slope contrast at inventory
#  pixels. This is a smoke test, not a result: it runs before `observed` exists,
#  so it rasterises the inventory itself, cheaply and centre-only.
# ---------------------------------------------------------------------------
if gpd is not None and os.path.exists(os.path.join(INPUT_DIR, FILE_LSI)):
    from rasterio.features import rasterize as _rz
    from shapely.geometry import mapping as _mp
    _g = gpd.read_file(os.path.join(INPUT_DIR, FILE_LSI))
    _g = _g.set_crs(LSI_CRS, allow_override=True).to_crs(ref_crs)
    _o = _rz([(_mp(x), 1) for x in _g.geometry if x is not None], out_shape=ref_shape,
             transform=ref_transform, fill=0, dtype=np.uint8, all_touched=False)
    _s = aligned['slope']; _m = np.isfinite(_s)
    _ls, _bg = _s[_m & (_o == 1)].mean(), _s[_m & (_o == 0)].mean()
    print(f"\nRegistration check: mean slope at landslide pixels {_ls:.2f}° "
          f"vs background {_bg:.2f}°, contrast {_ls-_bg:.2f}°")
    if _ls - _bg < 18.0:
        print("  ⚠ contrast below 18° — expected ~20° when slope, grid and inventory")
        print("    are mutually registered. Check GRID_CRS / LSI_CRS in §0.")
    else:
        print("  ✓ consistent with correct registration")
    del _g, _o, _s, _m

# ---------------------------------------------------------------------------
#  Second check, for GRID_CRS specifically. The one above tests slope against
#  the inventory, which is insensitive to GRID_CRS: if both are read in the same
#  frame they agree whichever frame that is. This one pins the grid itself.
#  EBK produces a conditional-mean surface, so the interpolated value at a
#  sample site should reproduce that site's measured value. Sampling all five
#  layers at the GPS sites, the correct GRID_CRS gives the lower mismatch.
# ---------------------------------------------------------------------------
_xl = os.path.join(INPUT_DIR, "Makilala Geotechnical Data.xlsx")
if os.path.exists(_xl):
    import pandas as _pd
    from shapely.geometry import Point as _Pt
    _raw = _pd.read_excel(_xl, sheet_name='GEOTECH MAPUA', header=None)
    _d = _raw.iloc[2:].copy()
    _d.columns = [str(x).strip() for x in _raw.iloc[1].tolist()]
    _pairs = [('sand', 'Sand %'), ('fines', 'Silt/Clay %'), ('pi', 'PI'),
              ('water', 'WATER CONTENT (%)'), ('gamma', 'BULK UNIT WEIGHT (kN/m3)')]
    for _c in ['LATITUDE', 'LONGITUDE'] + [b for _, b in _pairs]:
        _d[_c] = _pd.to_numeric(_d[_c], errors='coerce')
    _d = _d.dropna(subset=['LATITUDE', 'LONGITUDE'] + [b for _, b in _pairs])
    _p = gpd.GeoDataFrame(_d, geometry=[_Pt(x, y) for x, y in
                          zip(_d['LONGITUDE'], _d['LATITUDE'])],
                          crs='EPSG:4326').to_crs(ref_crs)
    _tot, _n = 0.0, 0
    for _k, _col in _pairs:
        _e = []
        for _pt, _meas in zip(_p.geometry, _d[_col]):
            _c2, _r2 = ~ref_transform * (_pt.x, _pt.y)
            _r2, _c2 = int(_r2), int(_c2)
            if 0 <= _r2 < ref_shape[0] and 0 <= _c2 < ref_shape[1]:
                _val = aligned[_k][_r2, _c2]
                if np.isfinite(_val):
                    _e.append(abs(_val - _meas))
        if _e:
            _tot += np.mean(_e) / max(abs(_d[_col].mean()), 1e-9); _n += 1
    print(f"\nGrid check: interpolated vs measured at {len(_d)} GPS sites, "
          f"normalised error {_tot:.3f} over {_n} layers")
    if _tot > 0.65:
        print("  ⚠ above 0.65 — the EBK layers do not reproduce their own sample")
        print("    values. GRID_CRS is probably wrong; ~0.56 is expected when right.")
    else:
        print("  ✓ EBK layers reproduce their sample values; GRID_CRS consistent")
    del _raw, _d, _p
else:
    print("\nGrid check skipped: Makilala Geotechnical Data.xlsx not found")


## §4 Derived parameter fields

$\alpha$, $z$, $\gamma$ come from rasters; $\phi'$, $c'$, $m$ from the
correlations of §2; $k_h = 0.5\,\mathrm{PGA}/g$ after the standard
pseudo-static convention.

In [ ]:
alpha = np.clip(aligned.get('slope', np.full(ref_shape, 25.0)), 0.1, 80.0)
z_raw = aligned.get('depth', np.full(ref_shape, 2.0))
gamma = np.clip(aligned.get('gamma', np.full(ref_shape, 12.17)), GAMMA_W+0.1, 30.0)
phi = estimate_phi(aligned.get('pi', np.full(ref_shape, np.nan)),
                   aligned.get('sand', np.full(ref_shape, 85.0)),
                   aligned.get('fines', np.full(ref_shape, 10.0)))
c = estimate_c(aligned.get('fines', np.full(ref_shape, np.nan)),
               aligned.get('sand', np.full(ref_shape, 85.0)))
m = estimate_m(aligned.get('water', np.full(ref_shape, np.nan)))

pga = aligned.get('pga', np.full(ref_shape, 0.4))
pga_max = np.nanmax(pga[np.isfinite(pga) & (pga > 0)])
if pga_max > 100: pga_g = pga / 981.0
elif pga_max > 2: pga_g = pga / 9.81
else: pga_g = pga
k_h = np.clip(0.5 * pga_g, 0.0, 1.0)

# Bind the raw interpolated fields as names. Part II §E (point-level
# validation) and §F (k_c uncertainty) both need them, and they otherwise
# exist only inside `aligned`. Direct indexing rather than .get(): if §3 did
# not load one of these, a KeyError here is better than a silent default
# propagating into the validation.
pi    = aligned['pi']
sand  = aligned['sand']
fines = aligned['fines']
water = aligned['water']

for name, arr in [("α°", alpha), ("z(m)", z_raw), ("γ", gamma), ("φ′°", phi),
                   ("c′(kPa)", c), ("m", m), ("k_h", k_h)]:
    v = np.isfinite(arr) & (arr > 0) if name == "k_h" else np.isfinite(arr)
    print(f"  {name:10s}: [{np.nanmin(arr[v]):.2f}, {np.nanmax(arr[v]):.2f}], mean {np.nanmean(arr[v]):.2f}")


## §5 Landslide inventory

The PHIVOLCS shapefile is **province-wide**: 3,140 polygons, of which 1,306 fall
outside the raster extent entirely and 135 touch it but lie outside geotechnical
coverage. The validation set is the **1,699** within the municipal footprint.
Every denominator in the manuscript is that 1,699, or the 1,002 of them that
carry at least one scored pixel — never 3,140.

In [ ]:
lsi_path = os.path.join(INPUT_DIR, FILE_LSI)
lsi = gpd.read_file(lsi_path) if gpd and os.path.exists(lsi_path) else None

# The shapefile's .prj declares ESRI:102457 but its coordinates are WGS 84 /
# UTM 51N; see the CRS note in §0. set_crs() relabels without moving anything,
# to_crs() then puts it on whatever GRID_CRS says the reference grid is. When
# LSI_CRS == GRID_CRS this is a no-op and the geometry is untouched.
if lsi is not None:
    lsi = lsi.set_crs(LSI_CRS, allow_override=True).to_crs(ref_crs)
    print(f"Inventory read as {LSI_CRS}, placed on the reference grid ({ref_crs})")

# Rasterization is centre-only (all_touched=False): a cell counts as a landslide
# pixel only if its centre falls inside a polygon. all_touched=True was tested
# and rejected -- it dilates every polygon by one cell, inflating the positive
# class from 4.56 to 5.48 km^2 (+20%) while recovering only 9 polygons.
#
# NOTE the asymmetry, which the manuscript's methods section must state: pixels
# are LABELLED centre-only here, but pixels are ATTRIBUTED to polygons on a
# touched-cell basis in the validation cell (poly_id, all_touched=True), so that
# a polygon with no centre-hit cell can still be evaluated. The two conventions
# differ by 9 polygons and 0.7 pp of detection rate, and the quoted 75.9% uses
# the touched-cell attribution.
ALL_TOUCHED = False

if lsi is None:
    raise RuntimeError(f"LSI inventory not found at {lsi_path}")

print(f"LSI (province-wide): {len(lsi):,} polygons, "
      f"ESI levels {sorted(lsi['ESI_local'].unique())}")
print(f"  mapped area      : {lsi['area'].sum()/1e6:.2f} km^2")
print("  the next cell clips this to Makilala and builds observed / esi_rasters")

### §5.1 Clip to the municipal footprint, and the structure of the depth gap

Two things happen here, and the second is the more important finding.

The inventory is clipped to the geotechnical footprint, and `observed` and the
ESI rasters are rebuilt from that subset.

Then the soil-depth coverage is characterised. **Depth is the only binding
constraint on the evaluated domain**: $\gamma$, $\phi'$, $c'$ and $m$ cost
nothing, the $\alpha \ge 10^\circ$ filter costs 13 polygons, and absent depth
costs a further 684. The gaps are not randomly placed — coverage falls from 75%
below $10^\circ$ to 53–56% above, and **50.1% of mapped landslide pixels fall in
the gap**. The evaluated domain is therefore biased away from exactly the
terrain the model exists to assess, which is why the manuscript reports the
measured-depth domain explicitly rather than imputing across it.

`DEPTH_MODE` selects between reporting that domain (`'none'`, the published
choice), constant fill, and nearest-neighbour fill. Imputation was tested and
degrades AUC in every configuration; the comparison is in the manuscript.

In [ ]:
# ============================================================================
# Runs after the inventory load (Part I §5) and before the sweep (§6).
#
# Two corrections:
#   A. LSI_North Cotabato.shp is PROVINCE-wide. Its extent strictly contains the
#      geotechnical footprint, and 1,306 of 3,140 polygons fall entirely outside
#      the raster. Validation must be reported against the Makilala subset.
#   B. The SoilGrids depth raster covers 64.3% of the footprint, and the gaps are
#      the ONLY binding constraint on the scored domain (gamma, phi, c and m cost
#      nothing; the alpha >= 10 deg filter costs 13 polygons). The gaps are also
#      spatially biased: 25% missing below 10 deg, 44-47% above, and 48.9% on
#      landslide pixels. Build a gap-filled depth so the coverage effect can be
#      isolated from everything else.
# ============================================================================

PIXEL_M2 = abs(ref_profile['transform'].a * ref_profile['transform'].e)
print(f"Pixel: {abs(ref_profile['transform'].a):.1f} m  ({PIXEL_M2:.0f} m^2)")

# ---- A. Clip the inventory to the geotechnical footprint -------------------
covered = np.isfinite(gamma) & np.isfinite(alpha)
poly_id = rasterize(
    [(mapping(g), i + 1) for i, g in enumerate(lsi.geometry) if g is not None],
    out_shape=ref_shape, transform=ref_profile['transform'],
    fill=0, dtype=np.int32, all_touched=True)

n_cov_poly = np.bincount(poly_id.ravel(), weights=covered.ravel(),
                         minlength=len(lsi) + 1)[1:]
in_mak = n_cov_poly > 0
lsi_mak = lsi.loc[in_mak].copy()

print(f"\nInventory:")
print(f"  province file            : {len(lsi):,} polygons")
print(f"  within Makilala footprint: {len(lsi_mak):,} polygons  <- validation set")
print(f"  outside raster extent    : {int((np.bincount(poly_id.ravel(), minlength=len(lsi)+1)[1:] == 0).sum()):,}")
print(f"  touching but uncovered   : {int((~in_mak).sum()) - int((np.bincount(poly_id.ravel(), minlength=len(lsi)+1)[1:] == 0).sum()):,}")
print(f"  ESI levels (Makilala)    : {sorted(lsi_mak['ESI_local'].unique())}")
print(f"  mapped area (Makilala)   : {lsi_mak['area'].sum()/1e6:.2f} km^2 "
      f"of {lsi['area'].sum()/1e6:.2f} km^2 province-wide")

# Rebuild observed / ESI rasters from the Makilala subset only
shapes = [(mapping(g), 1) for g in lsi_mak.geometry if g is not None]
observed = rasterize(shapes, out_shape=ref_shape, transform=ref_profile['transform'],
                     fill=0, dtype=np.uint8, all_touched=ALL_TOUCHED)

esi_rasters = {}
for esi in sorted(lsi_mak['ESI_local'].unique()):
    sub = lsi_mak[lsi_mak['ESI_local'] == esi]
    esi_rasters[int(esi)] = rasterize(
        [(mapping(g), 1) for g in sub.geometry if g is not None],
        out_shape=ref_shape, transform=ref_profile['transform'],
        fill=0, dtype=np.uint8, all_touched=ALL_TOUCHED)

print(f"  rasterized               : {int(np.sum(observed == 1)):,} landslide pixels")

# ---- B. Gap-filled soil depth ---------------------------------------------
# Fill value at the thin end of the 1-5 m translational range. This is the
# CONSERVATIVE direction: through c'/(gamma z cos^2 alpha), thinner soil raises
# FoS, so the fill biases AGAINST predicting failure on the recovered pixels.
Z_FILL_M = 1.2

z_missing = covered & (~np.isfinite(z_raw) | (z_raw <= 0))
z_filled = np.where(z_missing, Z_FILL_M, z_raw)

n_cov = int(covered.sum())
print(f"\nSoil depth:")
print(f"  footprint                : {n_cov:,} px  ({n_cov*PIXEL_M2/1e6:.1f} km^2)")
print(f"  SoilGrids coverage       : {100*np.sum(covered & np.isfinite(z_raw) & (z_raw > 0))/n_cov:.1f}%")
print(f"  gaps filled at {Z_FILL_M} m     : {int(z_missing.sum()):,} px "
      f"({100*z_missing.sum()/n_cov:.1f}% of footprint)")

# Is the gap spatially biased toward steep terrain? This is the question that
# determines whether the coverage loss is benign or concentrated on the hazard.
for lo, hi in [(0, 10), (10, 20), (20, 30), (30, 90)]:
    band = covered & (alpha >= lo) & (alpha < hi)
    if band.sum():
        print(f"    alpha {lo:>2}-{hi:<2} deg        : "
              f"{100*np.sum(band & z_missing)/band.sum():>5.1f}% missing "
              f"({int(band.sum()):>10,} px)")

ls_band = covered & (observed == 1)
print(f"    on landslide pixels    : {100*np.sum(ls_band & z_missing)/max(ls_band.sum(),1):>5.1f}% missing")

# Expose both depth families to the sweep. Rerun §6 with USE_FILLED_DEPTH
# True and False to separate the coverage effect from everything else.

DEPTH_MODE = 'none'          # 'none' | 'const' | 'nn'

z_valid = covered & np.isfinite(z_raw) & (z_raw > 0)

if DEPTH_MODE == 'const':
    z_base = np.where(z_missing, Z_FILL_M, z_raw)
elif DEPTH_MODE == 'nn':
    from scipy.ndimage import distance_transform_edt
    d, (iy, ix) = distance_transform_edt(~z_valid, return_distances=True,
                                         return_indices=True)
    z_base = np.where(covered, np.where(z_valid, z_raw, z_raw[iy, ix]), np.nan)
    px = abs(ref_profile['transform'].a)
    print(f"  NN fill distance: median {np.median(d[z_missing])*px:.0f} m, "
          f"90th pct {np.percentile(d[z_missing], 90)*px:.0f} m, "
          f"max {d[z_missing].max()*px:.0f} m")
    print(f"  filled z mean {np.nanmean(z_base[z_missing]):.2f} m vs "
          f"measured {np.nanmean(z_raw[z_valid]):.2f} m")
else:
    z_base = np.where(z_valid, z_raw, np.nan) if EXCLUDE_ZERO_DEPTH else z_raw
    if EXCLUDE_ZERO_DEPTH:
        n_zero = int(np.sum(covered & np.isfinite(z_raw) & (z_raw <= 0)))
        print(f'  EXCLUDE_ZERO_DEPTH: dropped {n_zero:,} zero-depth px')

print(f"\nDEPTH_MODE = {DEPTH_MODE}")



# USE_FILLED_DEPTH = True
# z_base = z_filled if USE_FILLED_DEPTH else z_raw
# print(f"\nUSE_FILLED_DEPTH = {USE_FILLED_DEPTH}")

## §6 The sweep: cohesion × depth cap × threshold

Twelve configurations — four cohesion scenarios (baseline, $+5$ kPa, $\times 2$,
$+10$ kPa) against three depth caps (20 m, 5 m, 3 m). For each: apply the
$\alpha \ge 10^\circ$ filter, sweep $\theta$ over $[0.40, 2.00]$ in steps of
0.05, and record hit rate, false alarm rate, Youden index and the exact AUC.

**On the lower bound.** v6 swept from 0.80 and reported an optimum of exactly
0.80 — the left endpoint. That was a boundary artifact, not an interior maximum,
and every threshold-dependent number in the earlier draft inherited it.
Extending the grid down to 0.40 moves the baseline optimum to 0.65 and gives all
twelve configurations interior optima. Any configuration still peaking at an
endpoint is flagged `CENSORED`; none currently is.

AUC is computed exactly from the Mann–Whitney identity, so it is threshold-free
and immune to the objection that $\theta^{*}$ is fitted on the validation set.

*This is the slow cell.*

In [ ]:
# ============================================================================
# Cohesion x depth x threshold sweep.
#
# Changes made when this replaced the v6 sweep cell:
#   1. Threshold grid extended DOWN to 0.40 (was 0.80 -> theta* was censored
#      at the left endpoint, not an interior optimum)
#   2. Sweeps all 4 cohesion scenarios x 3 z-caps (was baseline cohesion only)
#   3. Reports exact AUC per configuration -- threshold-free, immune to the
#      "threshold fitted on the validation set" objection
#   4. Flags any configuration whose optimum still sits on a grid endpoint
#   5. Sweep done by sorting once + searchsorted, not per-threshold reductions
#
# Downstream cells 15/17/19 are unaffected: `all_results` keeps its original
# structure and keys, populated from the BASELINE cohesion scenario.
# ============================================================================

THRESHOLDS = np.arange(0.40, 2.05, 0.05)
# THRESHOLDS = np.arange(0.40, 1.55, 0.05)          # was np.arange(0.80, 1.55, 0.05)

z_scenarios = {
    'z_original': np.clip(z_base, 0.1, 20.0),
    'z_cap_5m':   np.clip(z_base, 0.1, 5.0),
    'z_cap_3m':   np.clip(z_base, 0.1, 3.0),
}
# z_scenarios = {
#     'z_original': np.clip(z_raw, 0.1, 20.0),
#     'z_cap_5m':   np.clip(z_raw, 0.1, 5.0),
#     'z_cap_3m':   np.clip(z_raw, 0.1, 3.0),
# }

c_scenarios = {
    'c_baseline': c,
    'c_plus5':    c + 5.0,
    'c_times2':   c * 2.0,
    'c_plus10':   c + 10.0,
}


def sweep_config(c_arr, z_arr, observed, alpha, gamma, phi, m, k_h):
    """Compute FoS for one (cohesion, depth) configuration and sweep thresholds.

    Returns a dict with the per-threshold table, the AUC, and diagnostics.
    """
    v = (np.isfinite(alpha) & (alpha >= ALPHA_MIN) &
         np.isfinite(z_arr)  & (z_arr > 0) &
         np.isfinite(gamma)  & (gamma > GAMMA_W) &
         np.isfinite(phi)    & (phi > 0) & (phi < 90.0) &
         np.isfinite(c_arr)  & (c_arr > 0) &
         np.isfinite(m)      & (m > 0) & (m < 1))

    fos = np.full(ref_shape, np.nan)
    fos[v] = compute_FoS(c_arr[v], phi[v], alpha[v], m[v], gamma[v], z_arr[v], k_h[v])

    fos_static = np.full(ref_shape, np.nan)
    fos_static[v] = compute_FoS(c_arr[v], phi[v], alpha[v], m[v], gamma[v], z_arr[v], 0.0)

    kc = np.full(ref_shape, np.nan)
    kc[v] = compute_kc(c_arr[v], phi[v], alpha[v], m[v], gamma[v], z_arr[v])

    # scored domain: valid AND FoS finite (drops pixels with missing PGA)
    scored = v & np.isfinite(fos)
    n_dropped = int(v.sum() - scored.sum())

    pos = np.sort(fos[scored & (observed == 1)])   # landslide pixels
    neg = np.sort(fos[scored & (observed == 0)])   # stable pixels
    n_pos, n_neg = len(pos), len(neg)

    # ---- exact AUC (Mann-Whitney U on the score -FoS, ties at 0.5) ---------
    lt = np.searchsorted(neg, pos, side='left')    # neg strictly below each pos
    le = np.searchsorted(neg, pos, side='right')
    concordant = (n_neg - le).sum()                # neg above pos -> correct
    ties = (le - lt).sum()
    auc = (concordant + 0.5 * ties) / (n_pos * n_neg)

    # ---- threshold sweep via searchsorted ---------------------------------
    rows = []
    for t in THRESHOLDS:
        tp = int(np.searchsorted(pos, t, side='left'))   # FoS < t among positives
        fp = int(np.searchsorted(neg, t, side='left'))
        fn, tn = n_pos - tp, n_neg - fp
        hr  = tp / n_pos if n_pos else 0.0
        far = fp / n_neg if n_neg else 0.0
        rows.append({'thresh': float(t), 'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn,
                     'hr': hr, 'far': far,
                     'prec': tp / (tp + fp) if (tp + fp) else 0.0,
                     'acc': (tp + tn) / (n_pos + n_neg),
                     'youden': hr - far,
                     'pct_fail': 100.0 * (tp + fp) / (n_pos + n_neg),
                     'fos': fos, 'fos_static': fos_static, 'kc': kc})

    js = np.array([r['youden'] for r in rows])
    i_best = int(np.nanargmax(js))
    censored = (i_best == 0) or (i_best == len(rows) - 1)

    return {'sweep': rows, 'best_thresh': rows[i_best]['thresh'],
            'best_youden': rows[i_best]['youden'], 'auc': auc,
            'censored': censored, 'valid': v, 'scored': scored,
            'n_valid': int(v.sum()), 'n_scored': int(scored.sum()),
            'n_dropped': n_dropped, 'n_pos': n_pos, 'n_neg': n_neg,
            'fos': fos, 'fos_static': fos_static, 'kc': kc,
            'c_mean': float(np.nanmean(c_arr[v])),
            'c_min': float(np.nanmin(c_arr[v])), 'c_max': float(np.nanmax(c_arr[v])),
            'z_mean': float(np.nanmean(z_arr[v])),
            'fos_mean': float(np.nanmean(fos[scored]))}


assert observed is not None, "LSI inventory required -- run Part I §5 first"

print("=" * 104)
print(f"COHESION x DEPTH x THRESHOLD SWEEP   (alpha >= {ALPHA_MIN} deg, "
      f"theta in [{THRESHOLDS[0]:.2f}, {THRESHOLDS[-1]:.2f}])")
print("=" * 104)

grid = {}
for cname, c_arr in c_scenarios.items():
    for zname, z_arr in z_scenarios.items():
        grid[(cname, zname)] = sweep_config(c_arr, z_arr, observed,
                                            alpha, gamma, phi, m, k_h)

# ---- headline table -------------------------------------------------------
print(f"\n{'cohesion':<12} {'z-cap':<12} {'c mean':>7} {'z mean':>7} {'FoS mn':>7} "
      f"{'theta*':>7} {'Youden':>7} {'HitR':>7} {'FAR':>7} {'Acc':>7} {'AUC':>7}  flag")
print("-" * 104)
for (cname, zname), r in grid.items():
    b = [s for s in r['sweep'] if s['thresh'] == r['best_thresh']][0]
    flag = "CENSORED" if r['censored'] else ""
    print(f"{cname:<12} {zname:<12} {r['c_mean']:>7.2f} {r['z_mean']:>7.2f} "
          f"{r['fos_mean']:>7.3f} {r['best_thresh']:>7.2f} {r['best_youden']:>7.3f} "
          f"{100*b['hr']:>6.1f}% {100*b['far']:>6.1f}% {100*b['acc']:>6.1f}% "
          f"{r['auc']:>7.4f}  {flag}")

# ---- diagnostics ----------------------------------------------------------
ref = grid[('c_baseline', 'z_cap_3m')]
print("\n" + "=" * 104)
print("DIAGNOSTICS  (baseline cohesion, z-cap 3 m)")
print("-" * 104)
print(f"  valid pixels (mask v)          : {ref['n_valid']:,}")
print(f"  scored pixels (v & FoS finite) : {ref['n_scored']:,}")
print(f"  dropped (non-finite k_h/FoS)   : {ref['n_dropped']:,}"
      f"  ({100*ref['n_dropped']/max(ref['n_valid'],1):.2f}% of valid)")
print(f"  landslide / stable pixels      : {ref['n_pos']:,} / {ref['n_neg']:,}")

m_fallback = int(np.sum(ref['scored'] & (m == 0.5)))
print(f"  m == 0.50 fallback pixels      : {m_fallback:,}"
      f"  ({100*m_fallback/max(ref['n_scored'],1):.2f}% of scored)")

fs = ref['fos_static']
n_stable = int(np.sum(ref['scored'] & np.isfinite(fs) & (fs > 1)))
print(f"  Form B defined (FoS_static > 1): {n_stable:,}"
      f"  ({100*n_stable/max(ref['n_scored'],1):.2f}% of scored)"
      f"   -> NaN elsewhere")

n_cens = sum(1 for r in grid.values() if r['censored'])
if n_cens:
    print(f"\n  !! {n_cens}/{len(grid)} configurations still peak at a grid endpoint."
          f"\n     Widen THRESHOLDS further before quoting theta*.")
else:
    print(f"\n  All {len(grid)} configurations have an interior optimum.")

# ---- backward compatibility for cells 15 / 17 / 19 ------------------------
all_results = {zn: grid[('c_baseline', zn)] for zn in z_scenarios}
cohesion_results = grid
print("\n'all_results' populated from baseline cohesion; full grid in 'cohesion_results'.")

## §7 The primary configuration, and validation at four scales

Pins `c_plus10 / z_cap_5m` and produces the four results the discussion rests
on: the confusion matrix; the ESI-stratified hit rate; polygon-level detection
under both attribution conventions; and the spatially blocked cross-validation.

**The blocked result is the one that governs how the maps may be read.** Pooled
metrics are computed over a strongly autocorrelated sample — a single 2 km
neighbourhood on the western flank contributes tens of thousands of
non-independent positives. Under 2 km blocking, within-block AUC falls from
0.779 to $0.642 \pm 0.119$ and the out-of-sample Youden index from 0.431 to
$0.188 \pm 0.184$, while the median out-of-sample threshold is unchanged at
1.05. The framework separates hazardous from non-hazardous terrain across a
municipality; it discriminates between adjacent slopes only moderately.

**On the ESI intervals.** The denominator is the number of polygons at each
intensity carrying at least one scored pixel, not all polygons at that
intensity: the hit rate is computed over scored pixels, so a polygon
contributing none of them contributes nothing to the numerator either. Under the
correct denominator no adjacent pair of intervals separates — the cell computes
and prints that check rather than leaving it to the eye. The trend across
ESI 6–8 is monotonic and may be reported as such; it is not resolved by the
intervals.

In [ ]:
# ============================================================================
# Final validation at the pinned primary configuration.
#
# Produces the four things still missing before the Discussion can be written:
#   1. Confusion matrix at the primary configuration  (replaces draft Table 7)
#   2. ESI-stratified hit rate with POLYGON-based CIs (replaces draft Table 8)
#   3. Polygon-level detection rate
#   4. Spatially blocked AUC + out-of-sample theta*   (new; answers the
#      autocorrelation and threshold-fitting objections)
#
# Blocking uses square tiles rather than barangays: no administrative raster is
# needed, block size is tunable, and tiles are the standard construction for
# spatial cross-validation in susceptibility mapping.
#
# ---------------------------------------------------------------------------
# REVISED 26 Aug 2026. Three changes, all in parts 2 and 3:
#
#   (a) The ESI interval denominator was n = ALL polygons at that ESI within
#       Makilala. But the hit rate h is computed over SCORED pixels only, and
#       just 1,002 of 1,699 polygons contribute any. Counting polygons that
#       contribute nothing to the numerator shrinks se and makes the intervals
#       too narrow. n is now the number of polygons at that ESI carrying at
#       least one scored pixel. Both counts are printed so the change is
#       auditable.
#
#   (b) Adjacent-ESI interval separation is now computed and printed rather
#       than eyeballed off the table. Under (a) NO adjacent pair separates,
#       so the "disjoint confidence intervals" claim cannot be made. The
#       defensible claim is monotonic increase across ESI 6-8.
#
#   (c) Polygon attribution uses poly_id, built with all_touched=True, while
#       `observed` is deliberately rasterized centre-only (§5). That
#       asymmetry is the whole difference between 1,002 and 993 scored
#       polygons, and between 75.9% and 75.2% detection. Both are now
#       reported side by side so the methods can state which is used and why.
#       This supersedes the standalone polygon-detection cell, which looped
#       over the province-wide `lsi` (3,140) and rasterized the full grid
#       12,560 times to compute the centre-only column below.
#
# NOTE on the interval construction generally: se = sqrt(h(1-h)/n_poly) uses a
# PIXEL-level rate h with a POLYGON count n, as a deliberately conservative
# effective sample size. Polygons differ greatly in size (113 px mean among
# scored), so a properly clustered interval would be wider again. Whichever is
# adopted, Section 4 must say so explicitly.
# ============================================================================

BEST_KEY   = ('c_plus10', 'z_cap_5m')
BLOCK_PX   = 400          # 400 px x 5 m = 2 km tiles
MIN_BLOCK  = 200          # min positives AND negatives for a block to be scored

best_r      = cohesion_results[BEST_KEY]
BEST_Z      = BEST_KEY[1]
best_thresh = best_r['best_thresh']
fos_best    = best_r['fos']
scored      = best_r['scored']

print("=" * 92)
print(f"PRIMARY CONFIGURATION: {BEST_KEY[0]} / {BEST_KEY[1]}")
print("=" * 92)
print(f"  theta* {best_thresh:.2f}   J {best_r['best_youden']:.3f}   "
      f"pooled AUC {best_r['auc']:.4f}")
print(f"  c_mean {best_r['c_mean']:.2f} kPa   z_mean {best_r['z_mean']:.2f} m   "
      f"scored {best_r['n_scored']:,} px "
      f"({best_r['n_scored']*PIXEL_M2/1e6:.1f} km^2)")

fail_best = np.where(scored, np.where(fos_best < best_thresh, 1.0, 0.0), np.nan)
mask = scored
y    = (observed == 1)

# ---- 1. Confusion matrix ---------------------------------------------------
tp = int(np.sum(mask &  y & (fail_best == 1)))
fn = int(np.sum(mask &  y & (fail_best == 0)))
fp = int(np.sum(mask & ~y & (fail_best == 1)))
tn = int(np.sum(mask & ~y & (fail_best == 0)))
hr, far = tp / (tp + fn), fp / (fp + tn)

print(f"\n  {'':<20}{'Pred. failure':>16}{'Pred. stable':>16}")
print(f"  {'Observed landslide':<20}{tp:>16,}{fn:>16,}")
print(f"  {'Observed stable':<20}{fp:>16,}{tn:>16,}")
print(f"\n  hit {100*hr:.1f}%   FAR {100*far:.1f}%   "
      f"prec {100*tp/max(tp+fp,1):.1f}%   acc {100*(tp+tn)/(tp+fn+fp+tn):.1f}%   "
      f"J {hr-far:.3f}")

# ---- polygon-level bookkeeping (needed by BOTH parts 2 and 3) --------------
# poly_id is all_touched=True (§5.1): a polygon claims every cell it
# touches. Restricting to `mask` leaves only its scored cells.
pid_scored = np.where(mask, poly_id, 0)
n_px  = np.bincount(pid_scored.ravel(), minlength=len(lsi) + 1)[1:]
n_fal = np.bincount(pid_scored.ravel(),
                    weights=np.where(mask & (fail_best == 1), 1.0, 0.0).ravel(),
                    minlength=len(lsi) + 1)[1:]
has = n_px > 0                       # polygon carries >= 1 scored pixel

# ESI label per polygon, indexed over the full province-wide lsi so that it
# aligns with poly_id. -1 marks polygons outside the Makilala footprint.
esi_of_poly = np.full(len(lsi), -1, dtype=int)
esi_of_poly[np.where(in_mak)[0]] = lsi_mak['ESI_local'].to_numpy().astype(int)

# ---- 2. ESI-stratified, CI on the SCORED polygon count ---------------------
print(f"\n{'ESI':>4} {'poly':>6} {'scored':>7} {'LS_all':>10} {'LS_scored':>10} "
      f"{'TP':>8} {'hit':>7} {'95% CI':>16}")
print("-" * 84)
ci = {}
for esi in sorted(esi_rasters.keys()):
    oe = esi_rasters[esi] == 1
    at_esi   = (esi_of_poly == esi)
    npoly    = int(at_esi.sum())              # all polygons at this ESI
    npoly_sc = int((at_esi & has).sum())      # those carrying scored pixels
    n_all = int(oe.sum())
    t = int(np.sum(mask & oe & (fail_best == 1)))
    f = int(np.sum(mask & oe & (fail_best == 0)))
    h = t / (t + f) if (t + f) else 0.0
    se = (h * (1 - h) / npoly_sc) ** 0.5 if npoly_sc else 0.0
    lo, hi = max(0.0, h - 1.96 * se), min(1.0, h + 1.96 * se)
    ci[esi] = (lo, hi, h, npoly_sc)
    print(f"{esi:>4} {npoly:>6} {npoly_sc:>7} {n_all:>10,} {t+f:>10,} {t:>8,} "
          f"{100*h:>6.1f}% [{100*lo:>5.1f},{100*hi:>5.1f}]")

# Does the trend actually separate? Compute it rather than eyeball it.
lv = sorted(ci)
print("\n  adjacent-ESI interval separation:")
for a, b in zip(lv, lv[1:]):
    gap = ci[b][0] - ci[a][1]
    verdict = (f"disjoint by {100*gap:.2f} pp" if gap > 0
               else f"OVERLAP by {-100*gap:.2f} pp")
    print(f"    ESI {a} -> {b}: {100*ci[a][2]:>5.1f}% -> {100*ci[b][2]:>5.1f}%   {verdict}")
mono = all(ci[a][2] <= ci[b][2] for a, b in zip(lv, lv[1:]))
mono68 = all(ci[a][2] <= ci[b][2] for a, b in zip([6, 7], [7, 8]) if a in ci and b in ci)
print(f"    monotonic across all ESI: {mono}   across 6-8: {mono68}")
print("    -> claim monotonicity only; do NOT claim disjoint intervals unless"
      " every pair above reads 'disjoint'.")

# ---- 3. Polygon-level detection -------------------------------------------
# Centre-only attribution reported alongside, because `observed` is rasterized
# centre-only and the two give different denominators.
pid_centre = rasterize(
    [(mapping(g), i + 1) for i, g in enumerate(lsi.geometry) if g is not None],
    out_shape=ref_shape, transform=ref_profile['transform'],
    fill=0, dtype=np.int32, all_touched=False)
pidc_scored = np.where(mask, pid_centre, 0)
n_px_c  = np.bincount(pidc_scored.ravel(), minlength=len(lsi) + 1)[1:]
n_fal_c = np.bincount(pidc_scored.ravel(),
                      weights=np.where(mask & (fail_best == 1), 1.0, 0.0).ravel(),
                      minlength=len(lsi) + 1)[1:]
has_c = n_px_c > 0

print(f"\n  polygons with >=1 scored pixel")
print(f"    all_touched=True  (used) : {int(has.sum()):,} of {len(lsi_mak):,} in Makilala")
print(f"    centre-only              : {int(has_c.sum()):,}"
      f"   (difference {int(has.sum() - has_c.sum())} polygons)")
print(f"\n  {'DETECT_FRAC':>12} {'all_touched':>14} {'centre-only':>14}")
for fr in (0.01, 0.10, 0.25, 0.50):
    d  = int(np.sum((n_fal[has]   / n_px[has])     >= fr))
    dc = int(np.sum((n_fal_c[has_c] / n_px_c[has_c]) >= fr))
    print(f"  {fr:>11.0%} {d:>7,} ({100*d/has.sum():>4.1f}%) "
          f"{dc:>7,} ({100*dc/has_c.sum():>4.1f}%)")

# ---- 4. Spatial block cross-validation ------------------------------------
rows, cols = np.indices(ref_shape)
block = (rows // BLOCK_PX) * (ref_shape[1] // BLOCK_PX + 1) + (cols // BLOCK_PX)

def auc_of(pos, neg):
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    pos, neg = np.sort(pos), np.sort(neg)
    lo = np.searchsorted(neg, pos, side='left')
    hi = np.searchsorted(neg, pos, side='right')
    return ((len(neg) - hi).sum() + 0.5 * (hi - lo).sum()) / (len(pos) * len(neg))

bid  = block[mask]
fv   = fos_best[mask]
yv   = y[mask]
uniq = np.unique(bid)

per_block, kept = [], []
for b in uniq:
    s = bid == b
    p, n = fv[s & yv], fv[s & ~yv]
    if len(p) >= MIN_BLOCK and len(n) >= MIN_BLOCK:
        per_block.append(auc_of(p, n))
        kept.append(b)
per_block = np.array(per_block)

print(f"\n  Spatial blocks: {BLOCK_PX*5/1000:.1f} km tiles, "
      f"{len(kept)} of {len(uniq)} scored (>= {MIN_BLOCK} px per class)")
print(f"    pooled AUC          : {best_r['auc']:.4f}")
print(f"    within-block AUC    : {np.nanmean(per_block):.4f} "
      f"+/- {np.nanstd(per_block):.4f}   "
      f"[{np.nanmin(per_block):.3f}, {np.nanmax(per_block):.3f}]")

# Leave-one-block-out: theta* chosen on the other blocks, scored on the held-out
oos = []
for b in kept:
    tr, te = ~(bid == b), (bid == b)
    ptr, ntr = np.sort(fv[tr & yv]), np.sort(fv[tr & ~yv])
    js = [(np.searchsorted(ptr, t, 'left') / len(ptr)) -
          (np.searchsorted(ntr, t, 'left') / len(ntr)) for t in THRESHOLDS]
    th = THRESHOLDS[int(np.nanargmax(js))]
    pte, nte = fv[te & yv], fv[te & ~yv]
    h = np.mean(pte < th); fa = np.mean(nte < th)
    oos.append((th, h, fa, h - fa))
oos = np.array(oos)

print(f"    out-of-sample theta*: median {np.median(oos[:,0]):.2f} "
      f"[{oos[:,0].min():.2f}, {oos[:,0].max():.2f}]")
print(f"    out-of-sample hit   : {100*oos[:,1].mean():.1f}% "
      f"+/- {100*oos[:,1].std():.1f}%")
print(f"    out-of-sample FAR   : {100*oos[:,2].mean():.1f}% "
      f"+/- {100*oos[:,2].std():.1f}%")
print(f"    out-of-sample J     : {oos[:,3].mean():.3f} +/- {oos[:,3].std():.3f}"
      f"   (in-sample {best_r['best_youden']:.3f})")

## §8 Final fields

Binds the primary configuration's outputs to plain names and computes the Form B
bound, which is defined only where $\mathrm{FoS}_{\mathrm{static}} > 1$ — 82.2%
of the scored domain at this configuration. The undefined 17.8% is not scattered:
no scored pixel below $30^\circ$ is affected, against 46.2% of the $30$–$35^\circ$
band and 97.8% above $35^\circ$. The bound is undefined precisely on the steepest
ground, which the three-zone reading of the map must acknowledge.

In [ ]:
# Set final variables
fos = fos_best
fos_static = best_r['fos_static']
kc = best_r['kc']
valid = mask                 # pixel mask from §7; Part I must not depend on Part II
failure = fail_best
ac = np.where(kc > 0, G_ACC * kc, np.nan)
sens = np.full(ref_shape, np.nan)
stable = valid & np.isfinite(fos_static) & (fos_static > 1)
sens[stable] = sensitivity_FormB(fos_static[stable])

# Best confusion matrix
bst = [s for s in best_r['sweep'] if s['thresh'] == best_thresh][0]
print(f"FINAL MODEL: {BEST_Z}, FoS threshold={best_thresh:.2f}, α≥{ALPHA_MIN}°")
print(f"  Hit rate:    {100*bst['hr']:.1f}%")
print(f"  FAR:         {100*bst['far']:.1f}%")
print(f"  Precision:   {100*bst['prec']:.1f}%")
print(f"  Accuracy:    {100*bst['acc']:.1f}%")
print(f"  Youden:      {bst['youden']:.3f}")


## §9 Figures

The six-panel map (manuscript Fig. 3) and the validation overlay (Fig. 10).
Both are written to `OUTPUT_DIR` at 200 dpi.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
panels = [
    (alpha, 'Slope Angle α (°)', 'terrain', 0, 60),
    (fos_static, 'Static FoS', 'RdYlGn', 0.5, 3.0),
    (fos, f'FoS (Oct 2019)', 'RdYlGn', 0.5, 3.0),
    (kc, 'k_c', 'RdYlGn', -0.1, 0.5),
    (sens, '|S̃|_max', 'RdYlGn', 0, 1),
    (failure, f'Failure (FoS<{best_thresh:.2f}, α≥{ALPHA_MIN}°)', 'RdYlGn_r', 0, 1),
]
for ax, (data, title, cmap, vmin, vmax) in zip(axes.flat, panels):
    im = ax.imshow(np.where(valid, data, np.nan), cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
    ax.set_title(title, fontsize=11); plt.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle(f'Makilala — {BEST_Z}, thresh={best_thresh:.2f}', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'FoS_6panel.png'), dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Confusion overlay at the primary configuration.
# 0 = TN (grey), 1 = TP (green), 2 = FP (orange), 3 = FN (red), NaN elsewhere.
y = (observed == 1)
conf = np.full(ref_shape, np.nan)
conf[mask & ~y & (fail_best == 0)] = 0
conf[mask &  y & (fail_best == 1)] = 1
conf[mask & ~y & (fail_best == 1)] = 2
conf[mask &  y & (fail_best == 0)] = 3

obs_show  = np.where(mask, np.where(y, 1.0, 0.0), np.nan)
pred_show = np.where(mask, fail_best, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

axes[0].imshow(obs_show, cmap='Reds', vmin=0, vmax=1, aspect='equal')
axes[0].set_title(f'Observed landslide pixels\n({int((mask & y).sum()):,} px)', fontsize=11)

axes[1].imshow(pred_show, cmap='Reds', vmin=0, vmax=1, aspect='equal')
axes[1].set_title(f'Predicted failure (FoS < {best_thresh:.2f})\n'
                  f'({int(np.nansum(pred_show)):,} px)', fontsize=11)

cmap = ListedColormap(['#d9d9d9', '#2ca02c', '#ff9900', '#d62728'])
im = axes[2].imshow(conf, cmap=cmap, vmin=-0.5, vmax=3.5, aspect='equal')
tp = int((mask & y & (fail_best == 1)).sum()); fn = int((mask & y & (fail_best == 0)).sum())
fp = int((mask & ~y & (fail_best == 1)).sum()); tn = int((mask & ~y & (fail_best == 0)).sum())
axes[2].set_title(f'Confusion overlay\nTP {tp:,}  FN {fn:,}  FP {fp:,}  TN {tn:,}', fontsize=11)
cb = plt.colorbar(im, ax=axes[2], ticks=[0, 1, 2, 3], shrink=0.75)
cb.ax.set_yticklabels(['TN', 'TP', 'FP', 'FN'])

# --- legends, after the imshow calls -------------------------------------
axes[0].legend(handles=[
    Patch(facecolor='#67000d', label='Landslide'),
    Patch(facecolor='white', edgecolor='grey', label='Stable (scored)')],
    loc='lower left', fontsize=8, framealpha=0.9)

axes[1].legend(handles=[
    Patch(facecolor='#67000d', label='Predicted failure'),
    Patch(facecolor='white', edgecolor='grey', label='Predicted stable')],
    loc='lower left', fontsize=8, framealpha=0.9)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'Makilala — {BEST_KEY[0]} / {BEST_KEY[1]}, '
             f'$\\theta^*$ = {best_thresh:.2f}   '
             f'hit {100*tp/(tp+fn):.1f}%  FAR {100*fp/(fp+tn):.1f}%  '
             f'AUC {best_r["auc"]:.3f}', fontsize=13, y=0.99)
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, 'validation_fig6.png')
plt.savefig(out, dpi=200, bbox_inches='tight')
print("saved:", out)
plt.show()

## §10 Save rasters

In [ ]:
def save_raster(fp, data, profile):
    out = profile.copy(); out.update(dtype='float64', count=1, nodata=NODATA, compress='lzw')
    with rasterio.open(fp, 'w', **out) as dst:
        dst.write(np.where(np.isfinite(data), data, NODATA), 1)

for fn, d in {'FoS_map.tif': fos, 'FoS_static_map.tif': fos_static,
              'kc_map.tif': kc, 'ac_map.tif': ac,
              'sensitivity_FormB_map.tif': sens, 'failure_prediction.tif': failure}.items():
    save_raster(os.path.join(OUTPUT_DIR, fn), d, ref_profile)
    print(f"  ✓ {fn}")
print(f"\nAll in: {OUTPUT_DIR}")


---
# Part II — Diagnostics and supporting analyses

Each section is independent of the others but assumes Part I has run in the same
kernel. These are not leftovers: §B, §C and §F each produce a number the
manuscript cites, and §D and §E document negative results that would otherwise
have to be rediscovered.

## §A ROC and Youden index across depth caps

Drawn at the **correlation-baseline** cohesion, where the optima fall at 0.60,
0.65 and 0.75 — not at the primary configuration, which peaks at 1.05. The
manuscript's caption says so explicitly; the figure would otherwise be read as
contradicting the headline threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'z_original': 'blue', 'z_cap_5m': 'orange', 'z_cap_3m': 'green'}

# Panel 1: ROC curve
for zn, r in all_results.items():
    hrs = [s['hr'] for s in r['sweep']]
    fars = [s['far'] for s in r['sweep']]
    axes[0].plot(fars, hrs, 'o-', label=zn, color=colors[zn], markersize=4)
    # Mark best threshold
    best = [s for s in r['sweep'] if s['thresh'] == r['best_thresh']][0]
    axes[0].plot(best['far'], best['hr'], '*', color=colors[zn], markersize=15, zorder=5)

axes[0].plot([0,1], [0,1], 'k--', alpha=0.3, label='Random')
axes[0].set_xlabel('False Alarm Rate', fontsize=12)
axes[0].set_ylabel('Hit Rate', fontsize=12)
axes[0].set_title('ROC Curves (★ = best Youden index)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_xlim(-0.02, 1.02); axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(True, alpha=0.3)

# Panel 2: Youden index vs threshold
for zn, r in all_results.items():
    threshs = [s['thresh'] for s in r['sweep']]
    youdens = [s['youden'] for s in r['sweep']]
    axes[1].plot(threshs, youdens, 'o-', label=zn, color=colors[zn], markersize=4)
    axes[1].axvline(r['best_thresh'], color=colors[zn], ls=':', alpha=0.5)

axes[1].set_xlabel('FoS Threshold', fontsize=12)
axes[1].set_ylabel('Youden Index (HR − FAR)', fontsize=12)
axes[1].set_title('Optimal FoS Threshold', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ROC_and_Youden.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/ROC_and_Youden.png")


## §B Why polygons are unscored

Attributes each inventory polygon to one of: outside the geotechnical footprint,
inside but entirely below the slope filter, no valid soil depth, or scored. This
is where the "13 polygons to the slope filter, 684 to depth" split comes from.

`poly_scored` is the polygon-level array. It was called `scored` in v7, which
shadowed the pixel mask from §7 and crashed §8.

In [ ]:
# ---- Why are polygons unscored? -------------------------------------------
# Separates three causes: outside the geotechnical raster footprint (not part
# of the study area at all), inside the footprint but below the slope filter,
# and inside the scored domain.
poly_id = rasterize(
    [(mapping(g), i + 1) for i, g in enumerate(lsi.geometry) if g is not None],
    out_shape=ref_shape, transform=ref_profile['transform'],
    fill=0, dtype=np.int32, all_touched=True)


covered = np.isfinite(gamma) & np.isfinite(alpha)
steep   = covered & (alpha >= ALPHA_MIN)

n_tot  = np.bincount(poly_id.ravel(), minlength=len(lsi) + 1)[1:]
n_cov  = np.bincount(poly_id.ravel(), weights=covered.ravel(), minlength=len(lsi) + 1)[1:]
n_stp  = np.bincount(poly_id.ravel(), weights=steep.ravel(),   minlength=len(lsi) + 1)[1:]
n_scr  = np.bincount(poly_id.ravel(), weights=mask.ravel(),    minlength=len(lsi) + 1)[1:]

out_fp   = (n_tot > 0) & (n_cov == 0)                    # outside study area
too_flat = (n_cov > 0) & (n_stp == 0)                    # in area, all gentle
partial  = (n_stp > 0) & (n_scr < n_tot) & (n_scr > 0)   # partly scored
poly_scored = n_scr > 0

print(f"  polygons total                      : {len(lsi):,}")
print(f"  outside geotechnical footprint      : {out_fp.sum():,} "
      f"({100*out_fp.sum()/len(lsi):.1f}%)  <- not part of study area")
print(f"  inside footprint, all pixels <{ALPHA_MIN}deg : {too_flat.sum():,} "
      f"({100*too_flat.sum()/len(lsi):.1f}%)  <- model scope limitation")
print(f"  scored (>=1 pixel)                  : {poly_scored.sum():,} "
      f"({100*poly_scored.sum()/len(lsi):.1f}%)")
print(f"    of which partially clipped        : {partial.sum():,}")
print(f"\n  landslide pixels: {int(n_tot.sum()):,} total, "
      f"{int(n_cov.sum()):,} in footprint, {int(n_scr.sum()):,} scored")
print(f"  mean polygon size: {n_tot.mean():.0f} px overall, "
      f"{n_tot[poly_scored].mean():.0f} px scored, {n_tot[out_fp].mean():.0f} px out-of-footprint")

In [ ]:
z_ok = np.isfinite(z_scenarios[BEST_Z]) & (z_scenarios[BEST_Z] > 0)
n_zok = np.bincount(poly_id.ravel(), weights=z_ok.ravel(), minlength=len(lsi)+1)[1:]

no_depth = (n_stp > 0) & (n_zok == 0)
other    = (n_tot > 0) & ~out_fp & ~too_flat & ~poly_scored & ~no_depth
print(f"  steep but no valid soil depth : {no_depth.sum():,}")
print(f"  unexplained                   : {other.sum():,}")
print(f"  zero-pixel polygons           : {(n_tot == 0).sum():,}")
print(f"  depth coverage over footprint : "
      f"{100*np.sum(z_ok & covered)/np.sum(covered):.1f}%")

In [ ]:
valid_best = mask          # pixel mask from Part I §7; `scored` is polygon-level above
zb = z_scenarios[BEST_Z]
cond = {
    'in_footprint': covered,
    'alpha>=min'  : covered & (alpha >= ALPHA_MIN),
    '+z_finite'   : covered & (alpha >= ALPHA_MIN) & np.isfinite(zb) & (zb > 0),
    '+gamma_ok'   : covered & (alpha >= ALPHA_MIN) & np.isfinite(zb) & (zb > 0)
                    & np.isfinite(gamma) & (gamma > GAMMA_W),
    '+phi_ok'     : covered & (alpha >= ALPHA_MIN) & np.isfinite(zb) & (zb > 0)
                    & np.isfinite(gamma) & (gamma > GAMMA_W) & np.isfinite(phi) & (phi > 0),
    '+c_ok'       : covered & (alpha >= ALPHA_MIN) & np.isfinite(zb) & (zb > 0)
                    & np.isfinite(gamma) & (gamma > GAMMA_W) & np.isfinite(phi) & (phi > 0)
                    & np.isfinite(c) & (c >= 0),
    '+m_ok'       : valid_best,
    '+FoS_finite' : mask,
}
print(f"  {'condition':<14} {'px':>12} {'polys>=1px':>12}")
for k, cm in cond.items():
    n = np.bincount(poly_id.ravel(), weights=cm.ravel(), minlength=len(lsi)+1)[1:]
    print(f"  {k:<14} {int(cm.sum()):>12,} {int((n>0).sum()):>12,}")

## §C Depth sensitivity

The primary cohesion against all three depth caps, at $\theta^{*} = 1.05$
throughout. Manuscript Fig. 4.

In [ ]:
import matplotlib.pyplot as plt

# Depth sensitivity at the primary cohesion (c'+10 kPa), three z-caps,
# all at the primary threshold theta* = 1.05.
COH = BEST_KEY[0]          # 'c_plus10'
caps = ['z_original', 'z_cap_5m', 'z_cap_3m']
labels = {'z_original': '$z$-cap 20 m', 'z_cap_5m': '$z$-cap 5 m', 'z_cap_3m': '$z$-cap 3 m'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for j, cap in enumerate(caps):
    r = cohesion_results[(COH, cap)]
    f, s = r['fos'], r['scored']

    fos_show = np.where(s, f, np.nan)
    im0 = axes[0, j].imshow(fos_show, cmap='RdYlGn', vmin=0.5, vmax=3.0, aspect='equal')
    axes[0, j].set_title(f"{labels[cap]}\nFoS mean {r['fos_mean']:.3f}", fontsize=11)
    plt.colorbar(im0, ax=axes[0, j], shrink=0.7)

    fail = np.where(s, (f < best_thresh).astype(float), np.nan)
    pct = 100 * np.nansum(fail) / s.sum()
    im1 = axes[1, j].imshow(fail, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='equal')
    axes[1, j].set_title(f"Failure predicted ({pct:.1f}% of domain)\n"
                         f"$J$ = {r['best_youden']:.3f},  AUC = {r['auc']:.4f}", fontsize=11)
    plt.colorbar(im1, ax=axes[1, j], shrink=0.7)

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'Soil-depth sensitivity — {COH}, $\\theta^*$ = {best_thresh:.2f}',
             fontsize=14, y=0.98)
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, 'depth_sensitivity_6panel.png')
plt.savefig(out, dpi=200, bbox_inches='tight')
print("saved:", out)
plt.show()

## §D Error structure of the false negatives

Asks whether missed landslide pixels sit just above the threshold — in which
case the model is nearly right and a small recalibration would capture them — or
far above it, in which case they are being missed for a structural reason.
Includes the slope-angle comparison that tests the alternative explanation.

In [ ]:
fn_mask = mask & (observed == 1) & (fail_best == 0)
tp_mask = mask & (observed == 1) & (fail_best == 1)
fp_mask = mask & (observed == 0) & (fail_best == 1)

for nm, mk in [('FN', fn_mask), ('TP', tp_mask), ('FP', fp_mask)]:
    v = fos_best[mk]
    print(f"{nm}  n={mk.sum():>9,}  FoS median {np.nanmedian(v):.3f}  "
          f"IQR [{np.nanpercentile(v,25):.3f}, {np.nanpercentile(v,75):.3f}]")

# how close are the false negatives to the threshold?
v = fos_best[fn_mask]
for w in (0.05, 0.10, 0.25, 0.50):
    print(f"  FN within {w:.2f} above theta*: {100*np.mean(v < best_thresh + w):.1f}%")

# and their slope angles, to check the alternative explanation
print(f"\n  slope angle: FN median {np.nanmedian(alpha[fn_mask]):.1f} deg, "
      f"TP median {np.nanmedian(alpha[tp_mask]):.1f} deg")

In [ ]:
s = best_r['scored']
for nm, a in [('alpha', alpha), ('gamma', gamma), ('phi', phi), ('k_h', k_h)]:
    print(f"{nm:8s} {np.nanmin(a[s]):8.3f} {np.nanmax(a[s]):8.3f} {np.nanmean(a[s]):8.3f}")

## §E Point-level validation

**A documented negative result.** Point validation was attempted and is not
reported as a headline: the 43 sampled sites lie a median 74 m from the nearest
mapped polygon and only one falls inside one, so the sites characterise source
material rather than providing co-located labels.

The cell supersedes `Makilala_Point_Validation.ipynb`, which used a different
parameter scheme from this pipeline — a USCS lookup table plus a 1.3× cohesion
multiplier above 15% fines, neither of which appears in the manuscript. Here
everything calls this pipeline's own `estimate_c` / `estimate_phi` /
`estimate_m`, so the point-level and pixel-level validations describe the same
model. Two scorings are produced: raster-at-point (the map as deployed) and
sample-direct (the correlations without the interpolation step); the difference
between them is what interpolation costs at the only places where truth is known.

Reads all 43 collected samples, not the 37 with complete laboratory
determinations, because proximity does not depend on those fields.

In [ ]:
# ============================================================================
#  HARMONISED POINT-LEVEL VALIDATION
#
#  Replaces Makilala_Point_Validation.ipynb, which used a different parameter
#  scheme from the raster pipeline (a USCS lookup table plus a 1.3x multiplier
#  on cohesion where fines > 15%, neither documented in the manuscript). All
#  parameter estimation here calls the pipeline's own estimate_c(fines, sand),
#  estimate_phi(pi, sand, fines) and estimate_m(w), so the point-level and
#  pixel-level validations describe the same model.
#
#  Two scorings, answering different questions:
#    A. RASTER-AT-POINT  -- sample the FoS raster at each field site. Tests the
#       map as deployed.
#    B. SAMPLE-DIRECT    -- compute FoS from the site's own measured index
#       properties through the same correlations, with slope, depth and PGA
#       still from rasters (no point equivalent exists). Tests the correlations
#       without the interpolation step.
#
#  A vs B isolates what interpolation costs, at the only locations where the
#  ground truth is known.
#
#  Requires Part I to have completed.
# ============================================================================

import pandas as pd
import numpy as np
import rasterio
from rasterio.warp import transform as warp_transform

FIELD_XLSX = os.path.join(INPUT_DIR, "Makilala Geotechnical Data.xlsx")

df = pd.read_excel(FIELD_XLSX, header=1)
df.columns = [str(c).strip() for c in df.columns]
df = df[df['LATITUDE'].notna()].reset_index(drop=True)

LAT, LON, LABEL = 'LATITUDE', 'LONGITUDE', 'SITE TYPE'
FINES, SANDC, WATER, PI_C = 'Silt/Clay %', 'Sand %', 'WATER CONTENT (%)', 'PI'
BULK = [c for c in df.columns if 'BULK' in c.upper()][0]

print(f"field records: {len(df)}   (bulk column: '{BULK}')")
print(df[LABEL].value_counts().to_string())

# --- locate sites on the analysis grid -------------------------------------
xs, ys = warp_transform('EPSG:4326', ref_profile['crs'],
                        pd.to_numeric(df[LON], errors='coerce').tolist(),
                        pd.to_numeric(df[LAT], errors='coerce').tolist())
rr, cc = rasterio.transform.rowcol(ref_profile['transform'], xs, ys)
rr, cc = np.asarray(rr), np.asarray(cc)

in_grid = ((rr >= 0) & (rr < ref_shape[0]) & (cc >= 0) & (cc < ref_shape[1]))
rr_c, cc_c = np.clip(rr, 0, ref_shape[0] - 1), np.clip(cc, 0, ref_shape[1] - 1)
print(f"  inside grid extent: {int(in_grid.sum())} of {len(df)}")

def at_points(arr):
    out = np.full(len(df), np.nan)
    out[in_grid] = arr[rr_c[in_grid], cc_c[in_grid]]
    return out

# --- A. raster-at-point ----------------------------------------------------
fos_A = at_points(fos_best)
keep_A = in_grid.copy()
keep_A[in_grid] = mask[rr_c[in_grid], cc_c[in_grid]]

# --- B. sample-direct ------------------------------------------------------
alpha_pt = at_points(alpha)
z_pt     = at_points(z_scenarios[BEST_Z])
kh_pt    = at_points(k_h)

num = lambda col: pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
fines_pt, sand_pt = num(FINES), num(SANDC)
pi_pt, w_pt, g_pt = num(PI_C), num(WATER), num(BULK)

def safe(fn, *args):
    out = np.full(len(df), np.nan)
    for i in range(len(df)):
        vals = [a[i] for a in args]
        if all(np.isfinite(v) for v in vals):
            try:
                out[i] = float(fn(*vals))
            except Exception:
                pass
    return out

c_pt   = safe(estimate_c,   fines_pt, sand_pt)          # (fines, sand)
phi_pt = safe(estimate_phi, pi_pt, sand_pt, fines_pt)   # (pi, sand, fines)
m_pt   = safe(estimate_m,   w_pt)

# primary configuration applies a +10 kPa cohesion offset
c_pt_off = c_pt + 10.0

keep_B = (in_grid & np.isfinite(c_pt_off) & np.isfinite(phi_pt)
          & np.isfinite(m_pt) & np.isfinite(g_pt) & (g_pt > GAMMA_W)
          & np.isfinite(alpha_pt) & (alpha_pt >= ALPHA_MIN)
          & np.isfinite(z_pt) & (z_pt > 0) & np.isfinite(kh_pt))

fos_B = np.full(len(df), np.nan)
if keep_B.any():
    fos_B[keep_B] = compute_FoS(c_pt_off[keep_B], phi_pt[keep_B], alpha_pt[keep_B],
                                m_pt[keep_B], g_pt[keep_B], z_pt[keep_B], kh_pt[keep_B])

# --- scoring ---------------------------------------------------------------
obs = df[LABEL].astype(str).str.strip().str.lower().str.startswith('landslide').to_numpy()

def score(fos, keep, label):
    k = keep & np.isfinite(fos)
    if k.sum() == 0:
        print(f"\n  {label}: no scorable sites")
        return
    pred, y = fos[k] < best_thresh, obs[k]
    tp, fn = int((pred & y).sum()), int((~pred & y).sum())
    fp, tn = int((pred & ~y).sum()), int((~pred & ~y).sum())
    hr  = tp / (tp + fn) if (tp + fn) else np.nan
    far = fp / (fp + tn) if (fp + tn) else np.nan
    print(f"\n  {label}  (n={int(k.sum())}: {int(y.sum())} LS, {int((~y).sum())} non-LS)")
    print(f"    TP {tp}  FN {fn}  FP {fp}  TN {tn}")
    print(f"    hit {100*hr:.1f}%  FAR {100*far:.1f}%  "
          f"precision {100*tp/max(tp+fp,1):.1f}%  "
          f"accuracy {100*(tp+tn)/k.sum():.1f}%  J {hr-far:.3f}")
    print(f"    mean FoS: LS {np.nanmean(fos[k][y]):.3f}, "
          f"non-LS {np.nanmean(fos[k][~y]):.3f}")

print(f"\n  threshold theta* = {best_thresh:.2f}  ({BEST_KEY[0]} / {BEST_KEY[1]})")
score(fos_A, keep_A, "A. raster-at-point")
score(fos_B, keep_B, "B. sample-direct")

# --- what interpolation costs ---------------------------------------------
both = keep_A & keep_B & np.isfinite(fos_A) & np.isfinite(fos_B)
if both.sum() > 1:
    d = fos_B[both] - fos_A[both]
    flip = int(((fos_A[both] < best_thresh) != (fos_B[both] < best_thresh)).sum())
    print(f"\n  sites scorable both ways: {int(both.sum())}")
    print(f"    FoS difference (direct - raster): mean {np.mean(d):+.3f}  "
          f"sd {np.std(d):.3f}  max|d| {np.max(np.abs(d)):.3f}")
    print(f"    Spearman rank correlation: "
          f"{pd.Series(fos_A[both]).corr(pd.Series(fos_B[both]), method='spearman'):.3f}")
    print(f"    classified differently by the two routes: {flip} "
          f"({100*flip/both.sum():.0f}%)")

    print(f"\n  parameter values at sites, measured vs interpolated:")
    for nm, pt, ras in [('fines', fines_pt, fines), ('sand', sand_pt, sand),
                        ('water', w_pt, water), ('gamma', g_pt, gamma)]:
        try:
            rv = at_points(ras)
            k = both & np.isfinite(pt) & np.isfinite(rv)
            if k.sum():
                print(f"    {nm:<6} measured {np.nanmean(pt[k]):7.2f}  "
                      f"interpolated {np.nanmean(rv[k]):7.2f}  "
                      f"mean|diff| {np.nanmean(np.abs(pt[k]-rv[k])):6.2f}")
        except NameError:
            pass

In [ ]:
print(pd.DataFrame({'slope': alpha_pt, 'label': df[LABEL],
                    'scored': keep_A}).groupby(['label','scored'])['slope']
      .describe()[['count','mean','min','max']])

In [ ]:
print("landslide sites inside an inventory polygon:",
      int(observed[rr_c[in_grid], cc_c[in_grid]][obs[in_grid]].sum()),
      "of", int(obs[in_grid].sum()))

In [ ]:
for buf in [0, 1, 2, 5, 10, 20, 50]:
    hits = 0
    for i in np.where(obs & in_grid)[0]:
        r0, c0 = rr_c[i], cc_c[i]
        w = observed[max(0,r0-buf):r0+buf+1, max(0,c0-buf):c0+buf+1]
        hits += int((w == 1).any())
    print(f"  within {buf*5:>3} m: {hits}/25")

In [ ]:
from scipy import ndimage
d, (iy, ix) = ndimage.distance_transform_edt(observed == 0, return_indices=True)
sel = obs & in_grid
dy = (iy[rr_c[sel], cc_c[sel]] - rr_c[sel]) * 5
dx = (ix[rr_c[sel], cc_c[sel]] - cc_c[sel]) * 5
print(f"  north {np.mean(dy):+.0f} +/- {np.std(dy):.0f} m")
print(f"  east  {np.mean(dx):+.0f} +/- {np.std(dx):.0f} m")
print(f"  distance: median {np.median(np.hypot(dy,dx)):.0f} m")

## §F Uncertainty propagation for $k_c$

Answers Reviewer 2's point on parameter uncertainty using the paper's own closed
forms rather than a bolted-on Monte Carlo, treating the two parameter families
differently because they enter differently:

- **Continuous** ($\gamma$, $z$, $\alpha$, and $\phi'$ through PI): first-order
  delta method, $\mathrm{Var}(k_c) = \sum_i (\partial k_c/\partial x_i)^2
  \sigma_i^2$, with $\sigma$ from the EBK standard-error rasters.
- **Threshold-assigned** ($c'$ from fines, $m$ from water content): step
  functions, so not differentiable. The underlying variable is treated as
  Normal$(\mu, \sigma^2)$ and the class probabilities computed exactly.

That second treatment is the point of the exercise. The fines standard error is
~4.6 pp against class widths of 7–8 pp, so the cohesion class is genuinely
uncertain nearly everywhere.

**The standard-error rasters are reprojected, not zoomed.** They are EPSG:4326
in degrees over a ~16.5 km footprint; the reference grid is ESRI:102457 in
metres over 28.5 km. Reusing §3's `zoom` approach would stretch one footprint
across the other and misalign the errors by kilometres without raising anything.

Two caveats stated rather than assumed. These are the v1 realisation, while the
prediction rasters the pipeline reads match none of v1/v2/v3 exactly — run-to-run
variability is under 16% of the standard error, so this is second-order but not
zero. And the standard-error rasters cover only ~47.5% of the reference grid, so
the uncertainty map is defined on a subset of the scored domain.

*Needs roughly 3 GB beyond Part I's footprint; see the memory note at the top.*

In [ ]:
# ============================================================================
#  UNCERTAINTY PROPAGATION FOR k_c
#  Requires Part I to have completed: it needs `scored` and `best_r` from §7.
#
#  Answers Reviewer 2 point 4 using the paper's own closed forms rather than a
#  bolted-on Monte Carlo.
#
#  (A) CONTINUOUS parameters (gamma, z, alpha, phi' via PI): first-order delta
#      method, Var(k_c) = sum_i (dk_c/dx_i)^2 sigma_i^2, sigma from the EBK
#      prediction standard-error rasters.
#  (B) THRESHOLD-ASSIGNED parameters (c' from fines, m from water content):
#      step functions, not differentiable. The underlying continuous variable
#      is treated as Normal(mu, sigma^2) and the class probabilities computed
#      exactly, giving E[.] and Var[.] per pixel.
#
#  ---------------------------------------------------------------------------
#  CHANGES FROM kc_uncertainty_propagation.py, and why it could not run as-is:
#
#  1. `align_to_reference()` did not exist. The comment said "same helper as
#     the raster-load cell", but §3 has no such function -- it aligns inline
#     with scipy.ndimage.zoom inside a loop.
#
#  2. zoom() WOULD HAVE BEEN THE WRONG TOOL ANYWAY, and this is the important
#     one. The prediction rasters §3 aligns (Mak_*.tif) are already on the
#     reference CRS and extent, so a pure array rescale happens to be correct
#     for them. The standard-error rasters are NOT: they are EPSG:4326 in
#     degrees, ~74 m, spanning about 16.5 x 15.2 km, while the reference grid
#     is ESRI:102457 in metres, 5 m, spanning 28.5 x 18.5 km. zoom() would
#     stretch the smaller footprint across the larger one and misalign the
#     standard errors by kilometres, with no error raised. The helper below
#     reprojects properly.
#
#  3. `fines` and `water` were never bound as names. §4 keeps them in the
#     `aligned` dict and only derives c, m, phi from them.
#
#  4. Missing imports: os, rasterio, the warp helpers.
#
#  ---------------------------------------------------------------------------
#  OPEN PREREQUISITE, unchanged from the original header. These SE rasters are
#  the v1 realisation. The prediction rasters the pipeline actually reads
#  (Mak_*.tif) match none of v1/v2/v3 exactly and sit on a different grid, so
#  the pairing of v1 standard errors with those predictions is not exact.
#  Run-to-run variability is under 16% of SE (ebk_version_check_v4.py), so this
#  is second-order -- but it should be stated, not assumed away.
# ============================================================================

import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from scipy.stats import norm

SE_DIR = os.path.join(INPUT_DIR, "standard_error")

SE_FILES = {
    'gamma': 'buk_v1_standarderror.tif',
    'water': 'watercontent_v11standarderror.tif',
    'fines': 'siltclay_v1_standarderror.tif',
    'pi'   : 'PI_v1_standarderror.tif',
}


def align_to_reference(path):
    """Reproject a raster onto the reference grid.

    Not zoom(): these rasters are on a different CRS and a different extent
    from the reference grid. See note 2 in the header.
    """
    with rasterio.open(path) as src:
        a = src.read(1).astype(np.float32)
        if src.nodata is not None:
            a[a == src.nodata] = np.nan
        dst = np.full(ref_shape, np.nan, dtype=np.float32)
        reproject(source=a, destination=dst,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref_profile['transform'], dst_crs=ref_profile['crs'],
                  resampling=Resampling.bilinear,
                  src_nodata=np.nan, dst_nodata=np.nan)
    return dst


se = {}
for k, fn in SE_FILES.items():
    se[k] = align_to_reference(os.path.join(SE_DIR, fn))
    # EBK writes 0 where it has no prediction; that is nodata, not certainty.
    se[k] = np.where(se[k] > 0, se[k], np.nan)
    print(f"  SE {k:8s}: mean {np.nanmean(se[k]):.3f}  "
          f"[{np.nanmin(se[k]):.3f}, {np.nanmax(se[k]):.3f}]  "
          f"coverage {100*np.mean(np.isfinite(se[k])):.1f}% of grid")

# The continuous fields behind the two threshold rules (see note 3).
fines = aligned['fines']
water = aligned['water']


# ---------------------------------------------------------------------------
# (B) Threshold-assigned parameters -> class probabilities
# ---------------------------------------------------------------------------
def class_moments(mu, sigma, edges, values):
    """E[X] and Var[X] for a step function of a Normal variable."""
    mu = np.asarray(mu, float); sigma = np.asarray(sigma, float)
    cuts = [norm.cdf((e - mu) / sigma) for e in edges]
    probs = ([cuts[0]] +
             [cuts[i] - cuts[i - 1] for i in range(1, len(cuts))] +
             [1.0 - cuts[-1]])
    ex = sum(p * v for p, v in zip(probs, values))
    ex2 = sum(p * v * v for p, v in zip(probs, values))
    return ex, np.maximum(ex2 - ex ** 2, 0.0), probs


# cohesion from fines content (Das & Sobhan) -- matches estimate_c in §2
c_mean, c_var, c_probs = class_moments(
    fines, se['fines'], edges=[5.0, 12.0, 20.0], values=[1.0, 3.0, 6.0, 10.0])

# saturation from water content -- matches estimate_m in §2
m_mean, m_var, m_probs = class_moments(
    water, se['water'], edges=[20.0, 35.0, 50.0], values=[0.2, 0.4, 0.6, 0.8])

# --- match the cohesion scenario of the primary configuration ---------------
# class_moments returns the expectation of the BASELINE fines -> c' rule. The
# primary configuration applies a scenario offset on top of it (BEST_KEY[0]).
# Without this, k_c is computed at a different cohesion from everything else in
# the notebook: E[c'] came out at 3.53 kPa against the 13.38 kPa that §7 pins.
# An additive offset leaves the variance untouched; the multiplicative scenario
# scales it by the square.
_coh = BEST_KEY[0]
if   _coh == 'c_baseline': pass
elif _coh == 'c_plus5':    c_mean = c_mean + 5.0
elif _coh == 'c_plus10':   c_mean = c_mean + 10.0
elif _coh == 'c_times2':   c_mean, c_var = 2.0 * c_mean, 4.0 * c_var
else: raise ValueError(f"unhandled cohesion scenario {_coh!r}")

# the deterministic field at the same scenario, for a like-for-like comparison
c_det = {'c_baseline': c, 'c_plus5': c + 5.0,
         'c_times2': c * 2.0, 'c_plus10': c + 10.0}[_coh]

print(f"\n  cohesion scenario: {_coh}")
print(f"\n  E[c'] : mean {np.nanmean(c_mean[scored]):.2f} kPa   "
      f"sd {np.nanmean(np.sqrt(c_var[scored])):.2f}   "
      f"(deterministic assignment: {np.nanmean(c_det[scored]):.2f})")
print(f"  E[m]  : mean {np.nanmean(m_mean[scored]):.3f}   "
      f"sd {np.nanmean(np.sqrt(m_var[scored])):.3f}   "
      f"(deterministic: {np.nanmean(m[scored]):.3f})")

p_assigned = np.nanmax(np.stack(c_probs), axis=0)
print(f"  P(most likely c' class): mean {np.nanmean(p_assigned[scored]):.2f}, "
      f"below 0.5 on {100*np.nanmean(p_assigned[scored] < 0.5):.1f}% of pixels")

# ---------------------------------------------------------------------------
# (A) Delta method over the continuous parameters
# ---------------------------------------------------------------------------
# float32 throughout: at 21.1 M pixels a float64 copy is 169 MB and this block
# holds ~30 of them. See the memory note in the header.
f32 = np.float32
a   = np.radians(alpha).astype(f32)
ph  = np.radians(phi).astype(f32)
ca2 = (np.cos(a) ** 2).astype(f32)
Z   = z_scenarios[BEST_Z].astype(f32)
gamma_  = gamma.astype(f32)
c_mean  = c_mean.astype(f32);  c_var = c_var.astype(f32)
m_mean  = m_mean.astype(f32);  m_var = m_var.astype(f32)

d_c     = (1.0 / (gamma_ * Z * ca2)).astype(f32)
d_gamma = -c_mean / (gamma_ ** 2 * Z * ca2) + (m_mean * GAMMA_W / gamma_ ** 2) * np.tan(ph)
d_z     = (-c_mean / (gamma_ * Z ** 2 * ca2)).astype(f32)
d_m     = (-(GAMMA_W / gamma_) * np.tan(ph)).astype(f32)
d_phi   = ((1.0 - m_mean * GAMMA_W / gamma_) / np.cos(ph) ** 2).astype(f32)      # per radian
d_alpha = (2.0 * c_mean * np.tan(a) / (gamma_ * Z * ca2) - 1.0 / ca2).astype(f32)

# phi' = 35 - 0.1 PI (degrees) on the linear branch of estimate_phi. Where the
# USCS fallback applies, phi does not depend on PI and this overstates sigma.
sigma_phi = np.radians(0.1 * se['pi'])

SIGMA_Z_FRAC = 0.30              # SoilGrids depth, 30% relative -- assumption
SIGMA_ALPHA  = np.radians(2.0)   # DEM-derived slope, 2 degrees -- assumption

var_kc = (d_c ** 2     * c_var
        + d_gamma ** 2 * se['gamma'] ** 2
        + d_m ** 2     * m_var
        + d_phi ** 2   * sigma_phi ** 2
        + d_z ** 2     * (SIGMA_Z_FRAC * Z) ** 2
        + d_alpha ** 2 * SIGMA_ALPHA ** 2)

sd_kc = np.sqrt(var_kc)
kc_mu = (c_mean / (gamma_ * Z * ca2)
         + (1.0 - m_mean * GAMMA_W / gamma_) * np.tan(ph)
         - np.tan(a))

# ---------------------------------------------------------------------------
# Results
# ---------------------------------------------------------------------------
s = scored & np.isfinite(sd_kc) & np.isfinite(kc_mu)
print(f"\n  pixels retained: {int(s.sum()):,} of {int(scored.sum()):,} scored "
      f"({100*s.sum()/scored.sum():.1f}%)")
print(f"  k_c    : mean {np.nanmean(kc_mu[s]):.4f}  "
      f"[{np.nanpercentile(kc_mu[s], 5):.4f}, {np.nanpercentile(kc_mu[s], 95):.4f}]")
print(f"  sd(k_c): mean {np.nanmean(sd_kc[s]):.4f}  "
      f"median ratio sd/|k_c| = {np.nanmedian(sd_kc[s]/np.abs(kc_mu[s])):.2f}")
print(f"  pixels where sd(k_c) exceeds |k_c| : "
      f"{100*np.nanmean(sd_kc[s] > np.abs(kc_mu[s])):.1f}%")

# Variance shares are computed one term at a time and reduced immediately, so
# six more full-grid arrays are never resident together.
def _term(name):
    if name == 'c(fines)': return d_c**2 * c_var
    if name == 'gamma'   : return d_gamma**2 * se['gamma']**2
    if name == 'm(water)': return d_m**2 * m_var
    if name == 'phi(PI)' : return d_phi**2 * sigma_phi**2
    if name == 'z'       : return d_z**2 * (SIGMA_Z_FRAC*Z)**2
    if name == 'alpha'   : return d_alpha**2 * SIGMA_ALPHA**2
denom = np.nanmean(var_kc[s])
shares = {}
for _n in ('c(fines)','gamma','m(water)','phi(PI)','z','alpha'):
    _t = _term(_n); shares[_n] = 100*np.nanmean(_t[s])/denom; del _t
print("\n  variance share of k_c:")
for k, v in sorted(shares.items(), key=lambda kv: -kv[1]):
    print(f"    {k:10s} {v:>5.1f}%")

p_fail = norm.cdf((k_h - kc_mu) / sd_kc)
p_fail = np.where(s, p_fail, np.nan)
print(f"\n  P(k_c < k_h): mean {np.nanmean(p_fail[s]):.3f}")
print(f"    on landslide pixels : {np.nanmean(p_fail[s & (observed==1)]):.3f}")
print(f"    on stable pixels    : {np.nanmean(p_fail[s & (observed==0)]):.3f}")

pv, yv = p_fail[s], (observed == 1)[s]
pos, neg = np.sort(pv[yv]), np.sort(pv[~yv])
lo = np.searchsorted(neg, pos, 'left'); hi = np.searchsorted(neg, pos, 'right')
auc_p = ((hi - lo).sum() * 0.5 + (lo).sum()) / (len(pos) * len(neg))
print(f"  AUC of P(k_c < k_h)  : {auc_p:.4f}   "
      f"(deterministic FoS ranking: {best_r['auc']:.4f})")